# Chapter 11 — Building Agents: from Scratch and with LangGraph

Hands-On LangChain. Code targets **LangChain 1.x** / **LangGraph 1.x**.

This chapter builds agents two ways. First we implement a **ReAct agent from scratch** in plain Python —
no framework — so you see exactly how the reason/act loop works. Then we rebuild the same idea with
**LangGraph**, giving a real tool-using research agent backed by web search.

Companion notebook for Chapter 11. Set `MODEL_PROVIDER` to `ollama` or `groq` in `.env`; Groq chat requires `GROQ_API_KEY`.
The search agent uses **DuckDuckGo** (no API key required).

```bash
pip install langchain-ollama langchain-groq ollama groq langgraph langchain-community ddgs python-dotenv -q
```

## Part A — A ReAct agent from scratch

The **ReAct** (Reasoning + Acting) pattern, introduced in the 2022 paper *ReAct: Synergizing Reasoning
and Acting in Language Models*, has the model alternate between **thinking** and **doing**: it reasons
about what it needs, takes an action (calls a tool), observes the result, and repeats until it can
answer. We'll implement the loop ourselves to demystify it.

> Modern models have native tool/function calling that largely supersedes hand-built ReAct prompting for
> production — but building it by hand is the best way to understand what an agent actually does.

### A.1 A minimal agent class

In [1]:
import re
from model_provider import MODEL, MODEL_PROVIDER, get_raw_chat_completion

llm_model = MODEL
print("Using", MODEL_PROVIDER, "model:", llm_model)

class Agent:
    def __init__(self, system=""):
        self.system = system
        self.messages = []
        if self.system:
            self.messages.append({"role": "system", "content": system})

    def __call__(self, message):
        self.messages.append({"role": "user", "content": message})
        result = self.execute()
        self.messages.append({"role": "assistant", "content": result})
        return result

    def execute(self):
        return get_raw_chat_completion(self.messages, temperature=0)

### A.2 The ReAct system prompt

The prompt tells the model to operate in a loop of **Thought → Action → PAUSE → Observation**, lists the
actions it may take, and gives one worked example so it follows the format.

In [2]:
prompt = """
You operate in a cycle of Thought, Action, PAUSE, and Observation.
At the end of the loop, you provide an Answer.

Use Thought to reason about the information needed to answer the question you were asked.
Use Action to perform one of the available actions, then return PAUSE.
Observation will be the result of performing those actions.

Your available actions are:

get_current_weather:
e.g., get_current_weather: New York
Fetches the current weather for the specified location.

get_forecast:
e.g., get_forecast: San Francisco, 3 days
Provides a weather forecast for the given location and time range.

convert_temperature:
e.g., convert_temperature: 75 F to C
Converts a temperature from Fahrenheit to Celsius or vice versa.

Example session:

Question: What's the current temperature in Paris in Celsius?
Thought: I need to get the current temperature in Paris and convert it to Celsius.
Action: get_current_weather: Paris
PAUSE

You will then be called again with this:

Observation: The current temperature in Paris is 77 F.

You continue:

Thought: I should convert 77 F to Celsius.
Action: convert_temperature: 77 F to C
PAUSE

You will then be called again with this:

Observation: 77 F is approximately 25 C.

You finally output:

Answer: The current temperature in Paris is 25 C.""".strip()

### A.3 The action functions (the agent's tools)

In [3]:
def get_current_weather(location):
    # Placeholder implementation for fetching weather.
    return f"The current temperature in {location} is 32 F."

def get_forecast(location, days):
    return f"The weather forecast for {location} over the next {days} days is sunny with mild temperatures."

def convert_temperature(conversion):
    try:
        value, unit_from, _, unit_to = conversion.split()
        value = float(value)
        if unit_from.upper() == "F" and unit_to.upper() == "C":
            return f"{value} F is approximately {(value - 32) * 5 / 9:.1f} C."
        elif unit_from.upper() == "C" and unit_to.upper() == "F":
            return f"{value} C is approximately {(value * 9 / 5) + 32:.1f} F."
        return "Invalid temperature conversion. Use 'F to C' or 'C to F'."
    except Exception as e:
        return f"Error in temperature conversion: {e}"

known_actions = {
    "get_current_weather": get_current_weather,
    "get_forecast": get_forecast,
    "convert_temperature": convert_temperature,
}

### A.4 Stepping through the loop by hand\n\nAsk a question and watch the agent's first **Thought** and **Action**:

In [ ]:
abot = Agent(prompt)
result = abot("What is the temperature in Helsinki?")
print(result)

The agent chose an action. We run that tool ourselves and feed the result back as an **Observation**:

In [ ]:
observation = get_current_weather("Helsinki")
print("Observation:", observation)
print()
print(abot(f"Observation: {observation}"))

### A.5 Automating the loop

Doing that by hand is tedious. This `query` function runs the loop automatically: call the model, parse
any `Action`, run the matching tool, feed back the `Observation`, and repeat until the model stops
emitting actions.

In [6]:
action_re = re.compile(r'^Action: (\w+): (.*)$')   # parse an Action line

def query(question, max_turns=5):
    bot = Agent(prompt)
    next_prompt = question
    for _ in range(max_turns):
        result = bot(next_prompt)
        print(result)
        actions = [action_re.match(a) for a in result.split('\n') if action_re.match(a)]
        if not actions:
            return                              # no action -> the agent answered
        action, action_input = actions[0].groups()
        if action not in known_actions:
            raise Exception(f"Unknown action: {action}: {action_input}")
        print(f" -- running {action} {action_input}")
        observation = known_actions[action](action_input)
        print("Observation:", observation)
        next_prompt = f"Observation: {observation}"

In [ ]:
query("What is the temperature in Helsinki?")

## Part B — The same agent with LangGraph

Our hand-rolled loop works, but we wrote all the plumbing ourselves. **LangGraph** gives us that loop as
a proper **graph**: nodes do work, edges route control, and a **conditional edge** decides whether to
call a tool or finish. It also handles state and supports modern **native tool calling** (so no more
text parsing) and parallel tool calls.

LangGraph maps our agent to three pieces:
- an **`llm`** node that calls the model,
- a **conditional edge** that checks whether the model requested a tool,
- an **`action`** node that runs the requested tool(s) and loops back to the `llm` node.

### B.1 The search tool and agent state

In [8]:
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from model_provider import get_chat_model
from langchain_community.tools import DuckDuckGoSearchResults

# DuckDuckGo needs no API key. Return structured results, up to 4 per query.
tool = DuckDuckGoSearchResults(num_results=4, output_format="list")
print(type(tool))
print("tool name:", tool.name)

<class 'langchain_community.tools.ddg_search.tool.DuckDuckGoSearchResults'>
tool name: duckduckgo_results_json


In [9]:
# The state is just a list of messages. Annotating with operator.add means new
# messages are APPENDED to the list (not replaced) as the graph runs.
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

### B.2 The agent graph

In [10]:
from langgraph.graph import StateGraph, START, END

class Agent:
    def __init__(self, model, tools, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_model)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")     # after running tools, go back to the model
        graph.add_edge(START, "llm")        # entry point
        self.graph = graph.compile()
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def exists_action(self, state: AgentState):
        last = state["messages"][-1]
        return len(last.tool_calls) > 0

    def call_model(self, state: AgentState):
        messages = state["messages"]
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        return {"messages": [self.model.invoke(messages)]}

    def take_action(self, state: AgentState):
        tool_calls = state["messages"][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t['name']} {t['args']}")
            if t["name"] not in self.tools:
                result = "bad tool name, retry"
            else:
                result = self.tools[t["name"]].invoke(t["args"])
            results.append(ToolMessage(tool_call_id=t["id"], name=t["name"], content=str(result)))
        print("Back to the model!")
        return {"messages": results}

In [11]:
system_prompt = """You are a smart research assistant. Use the search engine to look up information. You are allowed to make multiple calls (either together or in sequence). Only look up information when you are sure of what you want. If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = get_chat_model(temperature=0)
abot = Agent(model, [tool], system=system_prompt)

### B.3 Visualizing the graph\n\nLangGraph can render the graph it built. Here is its structure as a Mermaid diagram:

In [12]:
print(abot.graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	llm(llm)
	action(action)
	__end__([<p>__end__</p>]):::last
	__start__ --> llm;
	action --> llm;
	llm -. &nbsp;False&nbsp; .-> __end__;
	llm -. &nbsp;True&nbsp; .-> action;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



### B.4 Running the agent\n\nA single-step question — the agent searches, then answers from what it finds:

In [ ]:
messages = [HumanMessage(content="What is the weather in Helsinki today?")]
result = abot.graph.invoke({"messages": messages})
print("\n" + result["messages"][-1].content)

A question about **two** cities — modern models can issue **parallel** tool calls, so the agent
searches both at once:

In [ ]:
messages = [HumanMessage(content="What is the weather in Helsinki and Stockholm?")]
result = abot.graph.invoke({"messages": messages})
print("\n" + result["messages"][-1].content)

A **multi-step** question, where later parts depend on earlier answers — the agent searches,
reasons, and searches again as needed:

In [ ]:
query_text = ("Who won the last football World Cup, and in which country was it held? "
              "Answer both questions.")
result = abot.graph.invoke({"messages": [HumanMessage(content=query_text)]})
print("\n" + result["messages"][-1].content)

> **The shortcut.** We built the graph by hand to show how it works. In practice LangGraph ships a
> prebuilt agent — `from langgraph.prebuilt import create_react_agent` — that constructs exactly this
> reason/act loop for you in one call. Build it by hand once to understand it; reach for the prebuilt in
> real projects.

That completes the book: from prompts and chains, through RAG, to agents that reason, use tools, and act.